# The other styles

[Notebook 1](01-building-a-grid.ipynb) built a British cryptic. The same
machinery makes American and barred puzzles, and what is interesting is how
little has to change: one rule set, one library, and — for barred grids — a
single line deciding where a run of cells ends.

1. What actually differs between the three
2. British: the half-checked lattice
3. American: every letter checked
4. Barred: no blocks at all
5. Where the grids come from, which is the real difference
6. Getting them out again

In [1]:
import os
import sys
import time
from collections import Counter

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from crossword import library
from crossword.fill import Filler
from crossword.index import Index
from crossword.render import show
from crossword.words import load

entries = load("crossword/UKACD.txt")
scores = {}
with open("crossword/scores.txt", encoding="utf-8") as handle:
    for line in handle:
        if not line.startswith("#") and line.strip():
            word, value = line.split()
            scores[word] = float(value)
index = Index(entries, scores)

## 1. What differs

Nothing in the search knows which style it is working on. Everything reads
*slots* — maximal runs of cells — and *checked cells*, and those are derived
from the grid rather than stored. So a style is a rule set plus a supply of
patterns.

Here are the three rule sets, and every field where they disagree.

In [2]:
styles = ("british", "us", "barred")
sets = {name: library.rules_for(name) for name in styles}

fields = [f for f in vars(sets["british"])
          if len({getattr(sets[n], f) for n in styles}) > 1]

print(f"{'':28}" + "".join(f"{name:>12}" for name in styles))
for field in fields:
    values = [getattr(sets[name], field) for name in styles]
    shown = [f"{v:.2f}" if isinstance(v, float) else str(v) for v in values]
    print(f"  {field:26}" + "".join(f"{s:>12}" for s in shown))

same = [f for f in vars(sets["british"]) if f not in fields]
print(f"\nidentical in all three: {', '.join(same)}")

                                 british          us      barred
  min_entry_length                     3           3           4
  max_consecutive_unchecked            1           0           1
  min_checked_fraction              0.50        1.00        0.67
  checked_fraction_rounds_up       False       False        True

identical in all three: max_checked_fraction, max_checked_slack, alternating, min_checked, forbid_run_length_two, symmetry


Two numbers carry almost all of it. `min_checked_fraction` is how much of an
entry has to be crossed, and `max_consecutive_unchecked` is whether two
uncrossed letters may sit side by side. British says half and one; American
says all and none; barred says two thirds and one.

The odd field out is `checked_fraction_rounds_up`, which decides which way the
fraction rounds when it does not divide evenly. British rounds down and barred
rounds up, and both were read off real puzzles rather than argued: the
commonest nine-letter British entry is UCUCUCUCU, which has four checked
letters and would fail a rule demanding five, while no entry in a published
Mephisto or Azed is more than a third uncrossed, which a rounding-down rule
would allow.

## 2. British

Half the letters of each entry are checked, and in practice they alternate
strictly: an entry reads CUCUC or UCUCU, never two of a kind together. That
lattice is why a British grid has so few entries — about 30 in a 15x15 — and
why they are long.

In [3]:
def summarise(grid, rules, label):
    slots = grid.slots(rules.min_entry_length)
    checked = grid.checked_cells(rules.min_entry_length)
    white = grid.size ** 2 - len(grid.blocks)
    lengths = [s.length for s in slots]
    print(f"{label}")
    print(f"  {grid.size}x{grid.size}, {len(grid.blocks)} blocks, "
          f"{len(slots)} entries")
    print(f"  entry length: {min(lengths)} to {max(lengths)}, "
          f"mean {sum(lengths) / len(lengths):.1f}")
    print(f"  {len(checked)} of {white} letters checked "
          f"({len(checked) / white:.0%})")


british = library.load()[0]
summarise(british.grid(), sets["british"], "British cryptic, Guardian library")
show(british.grid())

British cryptic, Guardian library
  15x15, 65 blocks, 30 entries
  entry length: 4 to 11, mean 7.3
  58 of 160 letters checked (36%)


## 3. American

Every letter is checked. That single change cascades: with no unchecked cells
there is no lattice of blocks to hold the grid apart, so the blocks thin out,
the entries get shorter, and there are far more of them — around 74 against 30.

It is much harder to fill. A British entry is pinned at alternate letters and
free between them; an American entry is pinned along its whole length, and
every word must agree with every word crossing it at every letter.

In [4]:
american = next(p for p in library.load(style="us") if 36 <= len(p.blocks) <= 40)
summarise(american.grid(), sets["us"], "American, pre-1965 New York Times library")
show(american.grid())

American, pre-1965 New York Times library
  15x15, 36 blocks, 78 entries
  entry length: 3 to 10, mean 4.8
  189 of 189 letters checked (100%)


Fill reliability depends steeply on how open the grid is, because a sparser
grid means longer entries and every letter of them is checked twice:

    blocks    filled    time
    43-49      9 / 10      5 s
    34         7 / 10     18 s
    23-27      2 / 10     48 s

Themed American grids do not work well yet. Seating chosen words into a fully
checked grid is much harder than into a British one, and every search parameter
in this project was tuned on 28-entry British grids.

## 4. Barred

No squares are blocked out at all. Every one of the 144 cells in a 12x12 holds
a letter, and the entries are separated by bars drawn *between* neighbours.

This is the only style needing a change to the grid code, and it is one
predicate. A run of cells ends at a block, at the edge, or now at a bar:

```python
bars = self.right_bars if direction == ACROSS else self.bottom_bars
...
if cell in bars:
    # The bar is after this cell, so the run includes it.
    found.append(Slot(...))
```

Everything downstream reads runs, so checkedness, the rules and the search
needed nothing. The unchecked letters arrive on their own: a run of length one
is a cell with a letter but no entry, and a cell whose across run is a single
is uncrossed in that direction.

In [5]:
from crossword.barred import parse

# A Mephisto, transcribed by hand. Geometry only: '_' marks a bar beneath a
# cell, '|' one between two cells. The grid is 180-degree symmetric, which is
# also the check that the transcription is right -- almost any slip breaks it.
MEPHISTO = '''* _ * * * _|* _ * * * *
* _ * * * * * _|*|*|*|*
* _ * * * * *|_ * * * *
* _ * * *|* * _ * * * *
*|*|_|*|_|*|*|_ _ * * *
_ * * * _ * * _ * * *|_
*|* * _ _ * * _ * _ * *
* * * * _|*|*|*|*|*|_|*
* * * * _ * *|* * * _ *
* * * * _|* * * * * _ *
*|*|*|*|_ * _ * * * _ *
* * * * * *|* * * * * *
'''

mephisto = parse(MEPHISTO)
summarise(mephisto, sets["barred"], "Mephisto, transcribed from the paper")
show(mephisto, min_length=4)

Mephisto, transcribed from the paper
  12x12, 0 blocks, 36 entries
  entry length: 5 to 11, mean 6.7
  96 of 144 letters checked (67%)


A third of the grid is uncrossed — 48 of 144 cells here, and 54 in the Azed
transcribed alongside it. That is far looser than it looks, and it is the
number to build to. A barred pattern leaving only a tenth of its cells
uncrossed satisfies every rule above and cannot be filled at all: the entries
cross each other so densely that no assignment of words survives.

The single cells are what produce those unches, and they cannot be scattered
freely. A cell that is single in *both* directions belongs to no entry at all,
so where a row puts its single cells decides where every column may put its
own.

In [6]:
runs = mephisto.runs("across") + mephisto.runs("down")
print("run lengths in a real barred grid:")
for length, count in sorted(Counter(r.length for r in runs).items()):
    kind = "single cells (unchecked letters)" if length == 1 else "entries"
    print(f"  {length:2d}: {count:3d}   {kind}")

checked = mephisto.checked_cells(4)
worst = max((s.length - sum(1 for c in s.cells if c in checked)) / s.length
            for s in mephisto.slots(4))
print(f"\nno entry is more than {worst:.0%} unchecked")

run lengths in a real barred grid:
   1:  48   single cells (unchecked letters)
   5:  12   entries
   6:   8   entries
   7:   8   entries
   8:   4   entries
  11:   4   entries

no entry is more than 33% unchecked


## 5. Where the grids come from

This is the real difference between the styles, and it is not a rule at all.

Randomly generated patterns that satisfy every rule fill very badly. Published
ones fill immediately. Setters know things about where blocks and bars can go
that none of these predicates capture, so wherever published grids can be had,
they are used.

| style | patterns | source |
|---|---|---|
| British | 120 | geometry of 8,348 published Guardian cryptics |
| American | 2,500 | geometry of pre-1965 New York Times puzzles |
| barred | 200 | generated, then filtered by actually filling them |

Barred is the exception because there are only two published barred grids to
hand. So the patterns are generated — rows drawn at random, columns *searched*
to fit them — and then each one is filled once before it is allowed into the
library. A pattern that can be filled is filled in about a second; one that
cannot burns the whole node budget first, which makes the test cheap and
decisive.

The same filler, on the same word list, tells the two apart plainly.

In [7]:
barred_rules = sets["barred"]

made = parse(MEPHISTO)
worker = Filler(made, index, barred_rules, seed=0,
                node_budget=60000, commonness=0.0)
began = time.time()
print(f"published Mephisto: filled={worker.fill()} "
      f"in {worker.stats.nodes} nodes, {time.time() - began:.2f}s")

pattern = library.load(style="barred")[0]
grid = pattern.grid()
worker = Filler(grid, index, barred_rules, seed=0,
                node_budget=60000, commonness=3.0, aim=0.85)
began = time.time()
print(f"library pattern:    filled={worker.fill()} "
      f"in {worker.stats.nodes} nodes, {time.time() - began:.2f}s")
show(grid, min_length=4)

published Mephisto: filled=True in 1352 nodes, 0.36s
library pattern:    filled=True in 9204 nodes, 12.12s


## 6. Getting them out

British and American grids are written as ipuz, for Exet and most desktop
software, and as Exolve, a self-contained web page.

Barred grids are written as neither. No bar can be placed in either format, and
a barred grid written into one would be a well-formed file describing a
different puzzle: a full square of white cells with every entry running the
whole width. Both writers refuse one outright rather than produce that, and
`export.write_html` draws a page instead — the grid with its bars, both clue
lists, and enumerations taken from the setter's own spelling.

In [8]:
from crossword import export

for writer, name in ((export.to_ipuz, "ipuz"), (export.to_exolve, "Exolve")):
    try:
        writer(parse(MEPHISTO))
    except NotImplementedError as refusal:
        print(f"{name}: {refusal}")

export.write_html(parse(MEPHISTO), "out/mephisto-blank.html",
                  title="Mephisto", min_length=4, solution=False)
print("\nwrote out/mephisto-blank.html instead")

ipuz: ipuz export does not support barred grids: the format written here has no way to place a bar
Exolve: Exolve export does not support barred grids: the format written here has no way to place a bar

wrote out/mephisto-blank.html instead


## Next

- **[ninas and pangrams](05-ninas-and-pangrams.ipynb)**: constraints a setter
  adds on top of the style

Back to [building a grid](01-building-a-grid.ipynb),
[words and the index](02-words-and-the-index.ipynb), or
[the fill search](03-the-fill-search.ipynb).